# Entregable 1 — Datos listos y persistidos
**Práctica integradora · Temas 1 (manipulación de datos) y 4 (bases de datos)**

Flujo: exploración inicial → limpieza y transformación → variables derivadas → diccionario y bitácora → esquema SQLite → carga → verificación.

> **Reproducibilidad:** ejecuta las celdas en orden. El notebook regenera `data/empleados_limpio.csv`, `data/empleados.db`, `docs/diccionario_datos.md` y `docs/bitacora_limpieza.md`. No usa aleatoriedad y la fecha de corte es fija.

In [1]:
import sqlite3, hashlib, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_CSV    = ROOT / "data" / "empleados_analitica.csv"
CLEAN_CSV  = ROOT / "data" / "empleados_limpio.csv"
DB_PATH    = ROOT / "data" / "empleados.db"
SCHEMA_SQL = ROOT / "sql" / "schema.sql"
DOCS       = ROOT / "docs"
FECHA_CORTE = pd.Timestamp("2025-06-30")   # fija, para que antigüedad sea reproducible
print("Raíz del proyecto:", ROOT)

Raíz del proyecto: /home/claude/practica-integradora-gab2402


## 1. Exploración inicial del CSV

In [2]:
raw = pd.read_csv(RAW_CSV)
print("Dimensiones:", raw.shape)
print("\nTipos de datos:"); print(raw.dtypes.to_string())
print("\nMuestra aleatoria (8 registros):"); print(raw.sample(8, random_state=0).to_string())

Dimensiones: (1250, 14)

Tipos de datos:
id_empleado                 str
edad                    float64
genero                      str
ciudad                      str
departamento                str
nivel_educativo             str
fecha_ingreso               str
anios_experiencia       float64
salario_mensual             str
horas_semanales         float64
num_capacitaciones      float64
evaluacion_desempeno    float64
satisfaccion            float64
abandono                  int64

Muestra aleatoria (8 registros):
     id_empleado  edad genero            ciudad      departamento nivel_educativo fecha_ingreso  anios_experiencia salario_mensual  horas_semanales  num_capacitaciones  evaluacion_desempeno  satisfaccion  abandono
711        E1613  40.0      F         Querétaro  Recursos Humanos    Licenciatura    2013-06-21               13.3         29253.4             42.9                 3.0                   1.0           5.9         0
898        E1370  36.0      M       Guadalajara  

In [3]:
nulos = pd.DataFrame({"nulos": raw.isna().sum(), "pct": (raw.isna().mean() * 100).round(1)})
print("Valores faltantes:"); print(nulos[nulos.nulos > 0].to_string())
print("\nFilas duplicadas exactas :", raw.duplicated().sum())
print("Ids repetidos (total)    :", raw["id_empleado"].duplicated().sum(),
      "-> duplicados parciales (mismo id, distinto contenido):",
      raw["id_empleado"].duplicated().sum() - raw.duplicated().sum())
print("Salarios con símbolo '$' (texto):", raw["salario_mensual"].astype(str).str.contains(r"\$").sum())
print("Fechas en formato dd/mm/aaaa     :", raw["fecha_ingreso"].astype(str).str.contains("/").sum())

Valores faltantes:
                      nulos  pct
edad                     25  2.0
ciudad                   12  1.0
nivel_educativo          74  5.9
anios_experiencia        63  5.0
salario_mensual          72  5.8
horas_semanales          40  3.2
num_capacitaciones       61  4.9
evaluacion_desempeno     50  4.0
satisfaccion             90  7.2

Filas duplicadas exactas : 30
Ids repetidos (total)    : 50 -> duplicados parciales (mismo id, distinto contenido): 20
Salarios con símbolo '$' (texto): 39
Fechas en formato dd/mm/aaaa     : 154


In [4]:
num = raw.copy()
num["salario_mensual"] = pd.to_numeric(num["salario_mensual"].astype("string").str.replace(r"[$,]", "", regex=True), errors="coerce")
print("Resumen numérico (salario ya convertido a número para esta inspección):")
print(num.describe().T[["min", "25%", "50%", "75%", "max"]].round(1).to_string())

print("\nValores fuera de rango evidentes:")
print("  edad < 18 o > 70            :", ((num.edad < 18) | (num.edad > 70)).sum())
print("  anios_experiencia < 0 o > 50:", ((num.anios_experiencia < 0) | (num.anios_experiencia > 50)).sum())
print("  horas_semanales < 20 o > 70 :", ((num.horas_semanales < 20) | (num.horas_semanales > 70)).sum())
print("  satisfaccion fuera de 1-10  :", ((num.satisfaccion < 1) | (num.satisfaccion > 10)).sum())
print("  salario < 4,000             :", (num.salario_mensual < 4000).sum())
print("  salario > 100,000           :", (num.salario_mensual > 100000).sum())

Resumen numérico (salario ya convertido a número para esta inspección):
                         min         25%       50%         75%        max
edad                     5.0        30.0      36.0        42.0      999.0
anios_experiencia       -3.0         4.2       9.0        13.5       75.0
salario_mensual       576.59  19919.5175  24852.01  32769.7875  382888.34
horas_semanales          0.0        40.8      44.0        47.3      200.0
num_capacitaciones       0.0         2.0       3.0         4.0       10.0
evaluacion_desempeno     1.0         3.0       3.0         4.0        5.0
satisfaccion             0.0         5.5       6.5         7.7       99.0
abandono                 0.0         0.0       0.0         0.0        1.0

Valores fuera de rango evidentes:
  edad < 18 o > 70            : 4
  anios_experiencia < 0 o > 50: 4
  horas_semanales < 20 o > 70 : 6
  satisfaccion fuera de 1-10  : 4
  salario < 4,000             : 5
  salario > 100,000           : 8


In [5]:
for c in ["genero", "ciudad", "departamento", "nivel_educativo"]:
    print(f"\n{c}:"); print(raw[c].value_counts(dropna=False).to_string())


genero:
genero
M           650
F           535
Femenino     65

ciudad:
ciudad
Ciudad de México      388
Monterrey             232
Guadalajara           206
Querétaro             109
Puebla                100
Mérida                 62
 Ciudad de México      21
CIUDAD DE MÉXICO       16
ciudad de méxico       15
guadalajara            14
NaN                    12
 Guadalajara           11
 Monterrey             10
GUADALAJARA             9
 Puebla                 6
MONTERREY               6
puebla                  5
 Mérida                 4
querétaro               4
monterrey               4
PUEBLA                  4
 Querétaro              4
QUERÉTARO               3
mérida                  3
MÉRIDA                  2

departamento:
departamento
Ventas                249
Tecnología            245
Operaciones           229
Logística             105
Marketing             101
Finanzas               95
Recursos Humanos       80
 Ventas                13
 Tecnología            13
VENTAS  

**Hallazgos de calidad:** duplicados exactos y por `id_empleado`; fechas en dos formatos; salario con texto (`$`, comas); categorías con mayúsculas/espacios inconsistentes (`genero` mezcla `F`/`Femenino`); valores imposibles (edad 5 o 999, experiencia negativa, satisfacción fuera de 1–10, horas 0 o 200); salarios extremos; y nulos en 9 columnas.

## 2. Pipeline de limpieza y transformación
Todo el tratamiento vive en una función `limpiar()` que devuelve el DataFrame limpio y una **bitácora** con el criterio y el número de filas afectadas por cada decisión.

**Orden de las operaciones** (importa): duplicados → tipos → categorías → rangos imposibles a nulo → imputación → coherencia entre variables → atípicos (IQR) → variables derivadas. Los valores imposibles se convierten primero en nulos para que se traten con la misma regla de imputación.

In [6]:
CATALOGO = {
    "ciudad": ["Ciudad de México", "Guadalajara", "Monterrey", "Puebla", "Querétaro", "Mérida"],
    "departamento": ["Ventas", "Tecnología", "Operaciones", "Finanzas", "Recursos Humanos", "Marketing", "Logística"],
    "nivel_educativo": ["Preparatoria", "Licenciatura", "Maestría", "Doctorado"],
}
SIN_CIUDAD = "No especificada"

def _norm(x):
    """minúsculas, sin acentos ni espacios sobrantes"""
    return unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode().strip().casefold()

def limpiar(raw):
    log = []
    def reg(paso, columna, criterio, filas):
        log.append({"paso": paso, "columna": columna, "criterio": criterio, "filas_afectadas": int(filas)})

    df = raw.copy()

    # --- 1. Duplicados -------------------------------------------------------
    n = df.duplicated().sum(); df = df.drop_duplicates()
    reg("1. Duplicados", "(fila completa)", "Se eliminan filas idénticas", n)
    df["_nulos"] = df.isna().sum(axis=1)
    n = df["id_empleado"].duplicated().sum()
    df = (df.sort_values(["id_empleado", "_nulos"], kind="stable")
            .drop_duplicates("id_empleado", keep="first").drop(columns="_nulos"))
    reg("1. Duplicados", "id_empleado", "Mismo id con contenido distinto: se conserva el registro con menos nulos", n)

    # --- 2. Normalización de tipos --------------------------------------------
    n = df["salario_mensual"].astype(str).str.contains(r"\$").sum()
    df["salario_mensual"] = pd.to_numeric(
        df["salario_mensual"].astype("string").str.replace(r"[$,\s]", "", regex=True), errors="coerce")
    reg("2. Tipos", "salario_mensual", "Texto con '$' y comas -> número (float)", n)

    f = df["fecha_ingreso"].astype("string")
    parsed = pd.to_datetime(f, format="%Y-%m-%d", errors="coerce")
    alt = pd.to_datetime(f, format="%d/%m/%Y", errors="coerce")
    n = parsed.isna().sum()
    df["fecha_ingreso"] = parsed.fillna(alt)
    assert df["fecha_ingreso"].notna().all(), "Hay fechas que no se pudieron interpretar"
    reg("2. Tipos", "fecha_ingreso", "Formatos mixtos (aaaa-mm-dd y dd/mm/aaaa) -> datetime", n)

    # --- 3. Categorías --------------------------------------------------------
    for col, lista in CATALOGO.items():
        mapa = {_norm(v): v for v in lista}
        antes = df[col].copy()
        df[col] = antes.map(lambda x: mapa.get(_norm(x), np.nan) if pd.notna(x) else np.nan)
        cambiadas = ((antes != df[col]) & antes.notna() & df[col].notna()).sum()
        sin_match = (antes.notna() & df[col].isna()).sum()
        reg("3. Categorías", col, "Mayúsculas/espacios/acentos -> etiqueta canónica del catálogo", cambiadas)
        assert sin_match == 0, f"{col}: {sin_match} valores fuera del catálogo"
    n = (df["genero"] == "Femenino").sum()
    df["genero"] = df["genero"].replace({"Femenino": "F"}).str.strip().str.upper()
    reg("3. Categorías", "genero", "'Femenino' -> 'F' (codificación única F/M)", n)

    # --- 4. Valores imposibles -> nulo ---------------------------------------
    reglas = {
        "edad":            (18, 70,   "Edad laboral legal/razonable 18-70"),
        "horas_semanales": (20, 70,   "Jornada plausible 20-70 h/semana"),
        "satisfaccion":    (1, 10,    "Escala de la encuesta 1-10"),
        "salario_mensual": (4000, None, "Piso plausible de 4,000 MXN (valores de 300-1,500 = error de captura)"),
    }
    for col, (lo, hi, crit) in reglas.items():
        malo = df[col].notna() & ((df[col] < lo) | ((df[col] > hi) if hi is not None else False))
        df.loc[malo, col] = np.nan
        reg("4. Rangos imposibles", col, crit + " -> se marca como nulo", malo.sum())

    # --- 5. Imputación --------------------------------------------------------
    def imputar(col, por=None, entero=False, motivo=""):
        mask = df[col].isna()
        fill = df.groupby(por)[col].transform("median") if por else pd.Series(df[col].median(), index=df.index)
        fill = fill.fillna(df[col].median())
        if entero: fill = fill.round()
        df.loc[mask, col] = fill[mask]
        reg("5. Imputación", col, motivo, mask.sum())
        return mask.astype(int)

    imputar("edad", "departamento", True, "Mediana por departamento (redondeada)")
    # coherencia: experiencia imposible (negativa, >50, o mayor que edad-16) -> nulo
    malo = df["anios_experiencia"].notna() & ((df.anios_experiencia < 0) | (df.anios_experiencia > 50) |
                                              (df.anios_experiencia > df.edad - 16))
    df.loc[malo, "anios_experiencia"] = np.nan
    reg("4. Rangos imposibles", "anios_experiencia", "Fuera de 0-50 o mayor que (edad-16) -> nulo", malo.sum())

    n = df["nivel_educativo"].isna().sum()
    moda = df["nivel_educativo"].mode()[0]
    df["nivel_educativo"] = df["nivel_educativo"].fillna(moda)
    reg("5. Imputación", "nivel_educativo", f"Moda global ({moda})", n)
    n = df["ciudad"].isna().sum(); df["ciudad"] = df["ciudad"].fillna(SIN_CIUDAD)
    reg("5. Imputación", "ciudad", f"Categoría explícita '{SIN_CIUDAD}' (no se inventa ubicación)", n)

    imputar("evaluacion_desempeno", None, True, "Mediana global (entera)")
    imputar("num_capacitaciones", None, True, "Mediana global (entera)")
    imputar("horas_semanales", "departamento", False, "Mediana por departamento")
    df["satisfaccion_imputada"]  = imputar("satisfaccion", "evaluacion_desempeno", False, "Mediana por evaluación de desempeño (variable relacionada)")
    df["experiencia_imputada"]   = imputar("anios_experiencia", "departamento", False, "Mediana por departamento")
    df["salario_imputado"]       = imputar("salario_mensual", ["departamento", "nivel_educativo"], False, "Mediana por departamento x nivel educativo")

    # coherencia posterior a imputar
    tope = np.maximum(df["edad"] - 16, 0)
    n = (df["anios_experiencia"] > tope).sum()
    df["anios_experiencia"] = np.minimum(df["anios_experiencia"], tope)
    reg("6. Coherencia", "anios_experiencia", "Tras imputar, se limita a (edad-16)", n)

    # --- 7. Atípicos de salario: IQR por departamento (winsorización) ----------
    g = df.groupby("departamento")["salario_mensual"]
    q1 = g.transform(lambda s: s.quantile(0.25)); q3 = g.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    lo, hi = (q1 - 1.5 * iqr).clip(lower=0), q3 + 1.5 * iqr
    nuevo = df["salario_mensual"].clip(lo, hi)
    df["salario_ajustado"] = (nuevo != df["salario_mensual"]).astype(int)
    df["salario_mensual"] = nuevo.round(2)
    reg("7. Atípicos", "salario_mensual", "IQR x 1.5 por departamento; se topa (winsoriza) en la valla, sin eliminar filas", df["salario_ajustado"].sum())

    # --- 8. Tipos finales y variables derivadas --------------------------------
    for c in ["edad", "num_capacitaciones", "evaluacion_desempeno", "abandono"]:
        df[c] = df[c].astype(int)
    df["anios_experiencia"] = df["anios_experiencia"].round(1)
    df["horas_semanales"]   = df["horas_semanales"].round(1)
    df["satisfaccion"]      = df["satisfaccion"].round(1)

    df["rango_edad"] = pd.cut(df["edad"], [17, 25, 35, 45, 55, 70], labels=["18-25", "26-35", "36-45", "46-55", "56-70"]).astype(str)
    df["antiguedad_anios"] = ((FECHA_CORTE - df["fecha_ingreso"]).dt.days / 365.25).clip(lower=0).round(1)
    df["alta_satisfaccion"] = (df["satisfaccion"] >= 8).astype(int)
    df["ratio_salario_depto"] = (df["salario_mensual"] / df.groupby("departamento")["salario_mensual"].transform("median")).round(3)
    df["banda_salarial"] = pd.cut(df["salario_mensual"], [0, 15000, 22000, 32000, np.inf],
                                  labels=["Baja", "Media", "Alta", "Muy alta"]).astype(str)
    for c, crit in [("rango_edad", "Cortes 18-25/26-35/36-45/46-55/56-70"),
                    ("antiguedad_anios", "(fecha de corte 2025-06-30 - fecha_ingreso) / 365.25"),
                    ("alta_satisfaccion", "1 si satisfaccion >= 8"),
                    ("ratio_salario_depto", "salario / mediana salarial del departamento"),
                    ("banda_salarial", "Cortes fijos: <=15k Baja, <=22k Media, <=32k Alta, >32k Muy alta")]:
        reg("8. Derivadas", c, crit, len(df))

    df["fecha_ingreso"] = df["fecha_ingreso"].dt.strftime("%Y-%m-%d")
    orden = ["id_empleado", "edad", "genero", "ciudad", "departamento", "nivel_educativo", "fecha_ingreso",
             "anios_experiencia", "salario_mensual", "horas_semanales", "num_capacitaciones",
             "evaluacion_desempeno", "satisfaccion", "abandono",
             "rango_edad", "antiguedad_anios", "alta_satisfaccion", "ratio_salario_depto", "banda_salarial",
             "salario_imputado", "experiencia_imputada", "satisfaccion_imputada", "salario_ajustado"]
    df = df[orden].sort_values("id_empleado").reset_index(drop=True)

    # --- Validaciones finales ---------------------------------------------------
    assert df["id_empleado"].is_unique
    assert df.drop(columns=[]).notna().all().all(), "Quedaron nulos"
    assert df["edad"].between(18, 70).all() and df["satisfaccion"].between(1, 10).all()
    assert df["anios_experiencia"].between(0, 50).all() and (df["anios_experiencia"] <= df["edad"] - 16).all()
    assert df["horas_semanales"].between(20, 70).all() and (df["salario_mensual"] > 0).all()
    return df, pd.DataFrame(log)

df, bitacora = limpiar(raw)
print("Filas: antes =", len(raw), "| después =", len(df))
print("Nulos restantes:", int(df.isna().sum().sum()), "| Duplicados restantes:", int(df.duplicated().sum()))
print("\nBitácora (resumen):"); print(bitacora.to_string(index=False))

Filas: antes = 1250 | después = 1200
Nulos restantes: 0 | Duplicados restantes: 0

Bitácora (resumen):
                paso              columna                                                                                    criterio  filas_afectadas
       1. Duplicados      (fila completa)                                                                 Se eliminan filas idénticas               30
       1. Duplicados          id_empleado                    Mismo id con contenido distinto: se conserva el registro con menos nulos               20
            2. Tipos      salario_mensual                                                     Texto con '$' y comas -> número (float)               39
            2. Tipos        fecha_ingreso                                       Formatos mixtos (aaaa-mm-dd y dd/mm/aaaa) -> datetime              150
       3. Categorías               ciudad                               Mayúsculas/espacios/acentos -> etiqueta canónica del catálogo         

In [7]:
print("Muestra del DataFrame limpio:"); print(df.head(6).to_string())
print("\nTipos finales:"); print(df.dtypes.to_string())
print("\nResumen del salario por departamento tras la limpieza:")
print(df.groupby("departamento")["salario_mensual"].agg(["count", "median", "min", "max"]).round(0).to_string())

Muestra del DataFrame limpio:
  id_empleado  edad genero            ciudad departamento nivel_educativo fecha_ingreso  anios_experiencia  salario_mensual  horas_semanales  num_capacitaciones  evaluacion_desempeno  satisfaccion  abandono rango_edad  antiguedad_anios  alta_satisfaccion  ratio_salario_depto banda_salarial  salario_imputado  experiencia_imputada  satisfaccion_imputada  salario_ajustado
0       E1000    39      M         Monterrey   Tecnología        Maestría    2019-06-10                9.0         63148.67             48.6                   0                     3           5.6         1      36-45               6.1                  0                1.611       Muy alta                 0                     0                      0                 1
1       E1001    27      M  Ciudad de México    Marketing    Licenciatura    2021-12-28                3.1          29518.3             54.7                   3                     4           2.4         1      26-35         

## 3. Diccionario de datos y bitácora de decisiones
Se generan como archivos Markdown en `docs/`. El diccionario se valida contra las columnas reales del DataFrame para que nunca quede desactualizado.

In [8]:
D = [  # (variable, tipo, descripción, rango / valores, notas de calidad)
 ("id_empleado", "TEXT (PK)", "Identificador único del empleado", "E1000-E2199", "Duplicados por id resueltos conservando el registro más completo."),
 ("edad", "INTEGER", "Edad en años cumplidos", "18-70", "Valores imposibles (5, 17, 150, 999) -> nulo -> mediana por departamento."),
 ("genero", "TEXT", "Género registrado", "F, M", "'Femenino' unificado a 'F'."),
 ("ciudad", "TEXT (FK)", "Ciudad de la sede", "Ciudad de México, Guadalajara, Monterrey, Puebla, Querétaro, Mérida, No especificada", "Formato normalizado; nulos -> 'No especificada'."),
 ("departamento", "TEXT (FK)", "Departamento al que pertenece", "Ventas, Tecnología, Operaciones, Finanzas, Recursos Humanos, Marketing, Logística", "Formato normalizado (mayúsculas/espacios)."),
 ("nivel_educativo", "TEXT", "Máximo nivel de estudios", "Preparatoria, Licenciatura, Maestría, Doctorado", "Nulos imputados con la moda."),
 ("fecha_ingreso", "TEXT (ISO aaaa-mm-dd)", "Fecha de ingreso a la organización", "2010-2025", "Dos formatos de origen unificados."),
 ("anios_experiencia", "REAL", "Años de experiencia laboral total", "0 - (edad-16), máx. 50", "Negativos y > edad-16 -> nulo -> mediana por departamento."),
 ("salario_mensual", "REAL", "Salario mensual bruto en MXN", "> 4,000 (acotado por IQR por departamento)", "Texto con '$' convertido; <4,000 -> imputado; extremos topados por IQR."),
 ("horas_semanales", "REAL", "Horas trabajadas por semana", "20-70", "Valores 0, 5, 99, 120, 150, 200 -> imputados por mediana de departamento."),
 ("num_capacitaciones", "INTEGER", "Cursos de capacitación en el último año", ">= 0", "Nulos -> mediana global."),
 ("evaluacion_desempeno", "INTEGER", "Evaluación de desempeño", "1-5", "Nulos -> mediana global."),
 ("satisfaccion", "REAL", "Satisfacción laboral (encuesta)", "1-10", "Valores 0, 11, 15, 99 -> nulo; imputada por mediana según desempeño."),
 ("abandono", "INTEGER", "1 si el empleado dejó la organización, 0 si continúa", "0, 1", "Variable objetivo candidata para el Entregable 3. Sin nulos en origen."),
 ("rango_edad", "TEXT", "DERIVADA: grupo etario", "18-25, 26-35, 36-45, 46-55, 56-70", "Calculada tras limpiar edad."),
 ("antiguedad_anios", "REAL", "DERIVADA: años en la organización a la fecha de corte 2025-06-30", ">= 0", "Fecha de corte fija para reproducibilidad."),
 ("alta_satisfaccion", "INTEGER", "DERIVADA: 1 si satisfaccion >= 8", "0, 1", "Indicador binario."),
 ("ratio_salario_depto", "REAL", "DERIVADA: salario / mediana salarial de su departamento", "> 0 (1 = en la mediana)", "Permite comparar entre departamentos."),
 ("banda_salarial", "TEXT", "DERIVADA: banda por cortes fijos de salario", "Baja, Media, Alta, Muy alta", "Cortes: 15k / 22k / 32k."),
 ("salario_imputado", "INTEGER", "MARCA: salario imputado (faltante o inválido)", "0, 1", "Útil para análisis de sensibilidad."),
 ("experiencia_imputada", "INTEGER", "MARCA: experiencia imputada", "0, 1", ""),
 ("satisfaccion_imputada", "INTEGER", "MARCA: satisfacción imputada", "0, 1", ""),
 ("salario_ajustado", "INTEGER", "MARCA: salario topado por la regla IQR", "0, 1", "Salario original extremo no se conserva; ver bitácora."),
]
assert [d[0] for d in D] == list(df.columns), "El diccionario no coincide con las columnas del DataFrame"

def md_tabla(filas, cab):
    esc = lambda x: str(x).replace("|", "\\|")
    return "\n".join(["| " + " | ".join(cab) + " |", "|" + "---|" * len(cab)] + ["| " + " | ".join(esc(c) for c in f) + " |" for f in filas])

DOCS.mkdir(exist_ok=True)
(DOCS / "diccionario_datos.md").write_text(
    "# Diccionario de datos — `empleados`\n\n" + md_tabla(D, ["Variable", "Tipo", "Descripción", "Valores / rango", "Notas de calidad"]) + "\n",
    encoding="utf-8")
(DOCS / "bitacora_limpieza.md").write_text(
    "# Bitácora de decisiones de limpieza\n\n"
    f"Registros de origen: **{len(raw)}** · Registros limpios: **{len(df)}**\n\n"
    + md_tabla(bitacora.values.tolist(), ["Paso", "Columna", "Criterio / decisión", "Filas afectadas"]) + "\n",
    encoding="utf-8")
df.to_csv(CLEAN_CSV, index=False, encoding="utf-8")
print("Escritos: docs/diccionario_datos.md, docs/bitacora_limpieza.md, data/empleados_limpio.csv")
print(pd.DataFrame(D, columns=["variable", "tipo", "descripcion", "rango", "notas"])[["variable", "tipo", "rango"]].to_string(index=False))

Escritos: docs/diccionario_datos.md, docs/bitacora_limpieza.md, data/empleados_limpio.csv
             variable                  tipo                                                                                rango
          id_empleado             TEXT (PK)                                                                          E1000-E2199
                 edad               INTEGER                                                                                18-70
               genero                  TEXT                                                                                 F, M
               ciudad             TEXT (FK) Ciudad de México, Guadalajara, Monterrey, Puebla, Querétaro, Mérida, No especificada
         departamento             TEXT (FK)    Ventas, Tecnología, Operaciones, Finanzas, Recursos Humanos, Marketing, Logística
      nivel_educativo                  TEXT                                      Preparatoria, Licenciatura, Maestría, Doctorado
       

## 4. Esquema SQLite (`sql/schema.sql`)
Modelo normalizado: **`ciudades`** y **`departamentos`** (catálogos) → **`empleados`** (hechos, con claves foráneas). Incluye restricciones `CHECK` que replican las reglas de limpieza, índices por departamento, ciudad, la combinación de ambos y `abandono`, y la vista `vw_empleados` con los nombres ya unidos.

In [9]:
print(SCHEMA_SQL.read_text(encoding="utf-8"))

-- Esquema relacional (SQLite) — Entregable 1
-- Modelo: 2 catálogos (ciudades, departamentos) + tabla de hechos (empleados).
-- Re-ejecutable: elimina y recrea los objetos.
PRAGMA foreign_keys = ON;

DROP VIEW  IF EXISTS vw_empleados;
DROP TABLE IF EXISTS empleados;
DROP TABLE IF EXISTS departamentos;
DROP TABLE IF EXISTS ciudades;

CREATE TABLE ciudades (
    id_ciudad INTEGER PRIMARY KEY,
    nombre    TEXT NOT NULL UNIQUE
);

CREATE TABLE departamentos (
    id_departamento INTEGER PRIMARY KEY,
    nombre          TEXT NOT NULL UNIQUE
);

CREATE TABLE empleados (
    id_empleado          TEXT    PRIMARY KEY,
    edad                 INTEGER NOT NULL CHECK (edad BETWEEN 18 AND 70),
    genero               TEXT    NOT NULL CHECK (genero IN ('F', 'M')),
    id_ciudad            INTEGER NOT NULL REFERENCES ciudades(id_ciudad),
    id_departamento      INTEGER NOT NULL REFERENCES departamentos(id_departamento),
    nivel_educativo      TEXT    NOT NULL CHECK (nivel_educativo IN ('Prepa

## 5. Carga de datos y verificación

In [10]:
if DB_PATH.exists():
    DB_PATH.unlink()            # se reconstruye desde cero: mismo resultado en cada ejecución
con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA foreign_keys = ON")
con.executescript(SCHEMA_SQL.read_text(encoding="utf-8"))
con.execute("PRAGMA foreign_keys = ON")

ciudades = sorted(df["ciudad"].unique()); deptos = sorted(df["departamento"].unique())
con.executemany("INSERT INTO ciudades (id_ciudad, nombre) VALUES (?, ?)", list(enumerate(ciudades, 1)))
con.executemany("INSERT INTO departamentos (id_departamento, nombre) VALUES (?, ?)", list(enumerate(deptos, 1)))

db = df.copy()
db["id_ciudad"] = db.pop("ciudad").map({n: i for i, n in enumerate(ciudades, 1)})
db["id_departamento"] = db.pop("departamento").map({n: i for i, n in enumerate(deptos, 1)})
cols = list(db.columns)
con.executemany(f"INSERT INTO empleados ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))})",
                db.astype(object).values.tolist())
con.commit()
print("Carga completada.")

Carga completada.


In [11]:
q = lambda sql: pd.read_sql_query(sql, con)

print("1) Conteos por tabla"); print(q("""
    SELECT 'empleados' AS tabla, COUNT(*) AS filas FROM empleados
    UNION ALL SELECT 'ciudades', COUNT(*) FROM ciudades
    UNION ALL SELECT 'departamentos', COUNT(*) FROM departamentos""").to_string(index=False))

print("\n2) Agregados por departamento"); print(q("""
    SELECT departamento, COUNT(*) AS empleados, ROUND(AVG(salario_mensual)) AS salario_prom,
           ROUND(AVG(satisfaccion), 2) AS satisfaccion_prom, ROUND(100.0 * AVG(abandono), 1) AS pct_abandono
    FROM vw_empleados GROUP BY departamento ORDER BY salario_prom DESC""").to_string(index=False))

print("\n3) Agregados por ciudad"); print(q("""
    SELECT ciudad, COUNT(*) AS empleados, ROUND(AVG(salario_mensual)) AS salario_prom,
           ROUND(AVG(horas_semanales), 1) AS horas_prom
    FROM vw_empleados GROUP BY ciudad ORDER BY empleados DESC""").to_string(index=False))

print("\n4) Departamento x banda salarial"); print(q("""
    SELECT departamento, banda_salarial, COUNT(*) AS n FROM vw_empleados
    GROUP BY departamento, banda_salarial ORDER BY departamento, MIN(salario_mensual)""").head(12).to_string(index=False))

1) Conteos por tabla
        tabla  filas
    empleados   1200
     ciudades      7
departamentos      7

2) Agregados por departamento
    departamento  empleados  salario_prom  satisfaccion_prom  pct_abandono
      Tecnología        264       40774.0               6.53          17.8
        Finanzas        103       31470.0               6.89          22.3
       Marketing        117       27829.0               6.44          13.7
Recursos Humanos         84       26095.0               6.52          29.8
          Ventas        272       22416.0               6.54          23.9
     Operaciones        242       21267.0               6.61          22.3
       Logística        118       19741.0               6.66          32.2

3) Agregados por ciudad
          ciudad  empleados  salario_prom  horas_prom
Ciudad de México        421       29720.0        43.9
       Monterrey        244       29249.0        44.2
     Guadalajara        230       26225.0        43.9
       Querétaro       

In [12]:
# Comprobaciones de integridad
assert q("SELECT COUNT(*) AS n FROM empleados")["n"][0] == len(df)
assert q("PRAGMA integrity_check")["integrity_check"][0] == "ok"
assert len(q("PRAGMA foreign_key_check")) == 0
assert q("SELECT COUNT(*) AS n FROM empleados WHERE id_ciudad NOT IN (SELECT id_ciudad FROM ciudades)")["n"][0] == 0
assert abs(q("SELECT SUM(salario_mensual) AS s FROM empleados")["s"][0] - df["salario_mensual"].sum()) < 1e-6
print("Integridad OK: filas = DataFrame, sin huérfanos en claves foráneas, suma de salarios coincide.")

print("\nÍndices creados:"); print(q("SELECT name AS indice, tbl_name AS tabla FROM sqlite_master WHERE type='index' AND name LIKE 'idx_%'").to_string(index=False))
con.close()

h = hashlib.sha256(CLEAN_CSV.read_bytes()).hexdigest()[:16]
print("\nHuella SHA-256 (16 car.) del CSV limpio:", h, "<- debe ser idéntica en cada ejecución")

Integridad OK: filas = DataFrame, sin huérfanos en claves foráneas, suma de salarios coincide.

Índices creados:
                     indice     tabla
       idx_emp_departamento empleados
             idx_emp_ciudad empleados
idx_emp_departamento_ciudad empleados
           idx_emp_abandono empleados

Huella SHA-256 (16 car.) del CSV limpio: d2cb8276ceb620e0 <- debe ser idéntica en cada ejecución


## 6. Conclusión del entregable
- Se pasó de **1,250** registros crudos a **1,200** empleados únicos, sin nulos y con tipos normalizados.
- Cada decisión (criterio y filas afectadas) quedó en `docs/bitacora_limpieza.md`; las variables, en `docs/diccionario_datos.md`.
- La base `data/empleados.db` se genera con `sql/schema.sql` y pasa las verificaciones de integridad.
- Las marcas `*_imputado` y `salario_ajustado` permiten medir en el Entregable 3 si la imputación afecta al modelo.